In [2]:
print('test')

test


In [3]:
#라이브러리 import
import pandas as pd   
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
import json

d:\dev\2026\js_fastApI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
# JSON 로드
with open("../storage/menus_recommendation.json", "r", encoding="utf-8") as f:
    data = json.load(f)
    
    # data 확인작업
print(data[:3])
type(data)   # 내가 준비한 파일을 로딩했는데 로딩된 결과가 어떤 타입
len(data) # 내가 몇 개의 정보를 읽어 왔는가?

[{'id': 5, 'name': '마파두부', 'calories': 400, 'description': '두부와 다진 고기를 매콤한 소스로 볶은 중국 요리', 'recommendation': 89}, {'id': 6, 'name': '깐풍기', 'calories': 750, 'description': '튀긴 닭고기를 매콤달콤한 소스에 볶은 요리', 'recommendation': 312}, {'id': 7, 'name': '양장피', 'calories': 500, 'description': '해산물과 채소를 겨자소스에 곁들인 중국 요리', 'recommendation': 194}]


97

In [5]:
# =========================
# 3. Document 생성
# =========================
## Documnet생성.. 왜 생성하느냐 이것으로 벡터베이터 베이스 만들것임
from langchain_core.documents import Document
## Document는 키가 미리 정의 되어 있어요.
## 키는  page_content,  metadata  //자바로 표현한다면 Document에 있는 변수

docs=[] #document를 저장하기 위한 리스트
for row in data:  # 로딩한 데이터를 하나씩 가져오겠다
    # print(row)
    # row가 딕셔너리이고 이 딕셔너리에서 다음과 같은 하나의 문장을 만든다.
    content = f"""
    이름:{row['name']}
    설명:{row['description']}
    """
    # 위 코드의 결과는 질문과 여행지와 설명의 키로 한 문장을 만든 것.
    # 의도가 이 문장을 벡터화 시켜서.. 새로운 문장과 유사한 문장을 찾으려는
    
    # RAG벡터를 만들때는 Document타입으로 합니다.
    doc = Document(
        page_content=content.strip()
        #metadata는 여기서는 사용하지 않습니다.
        #metadata는 임베딩에서 유사도 분석할 때는 사용하지 않고
        #현재 문서의 출처나 보조적인 설명을 추가 할 때 사용합니다.
        #이렇게 하면 답변의 출처나 신뢰가 높아 질 수 있다.
    )
    docs.append(doc)
print(docs[:2])

[Document(metadata={}, page_content='이름:마파두부\n    설명:두부와 다진 고기를 매콤한 소스로 볶은 중국 요리'), Document(metadata={}, page_content='이름:깐풍기\n    설명:튀긴 닭고기를 매콤달콤한 소스에 볶은 요리')]


In [ ]:
# =========================
# 4. 임베딩 + 벡터 DB
# 임베딩은 “텍스트를 의미 기반 숫자(벡터)로 바꿔서 비슷한 걸 찾는 기술”
# 문장을 숫자토큰으로 만든다.. 그리고 문장을 이해하기 위해서 벡터로 만든다.. 이정도로

# 1. 문서 → 임베딩 변환
# 2. 벡터 생성
# 3. FAISS 인덱스에 저장
# =========================
# docs
#  → (각 문서 content 꺼냄)
#  → embeddings.embed_documents()
#  → 벡터 리스트 생성
#  → FAISS 인덱스 생성
#  → vectorstore 객체 반환
 
## openai에서 제공하는 임베딩 모델을 사용해 본다.
## 실습을 하고 있고 성능보다는 개념 적인 접근
## 한글은 이 임베딩 모델의 성능 사용할만 하다.. 성능이 별로.. 한글은
## 임베딩의 핵심은 한글 문장에 이해도..
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
embeddings = OpenAIEmbeddings()
##  자연어를 토큰화 숫자를 만들어서(벡터) 문장을 의미있게 분석해서
##  의미있는 벡터를 만드는 것이 임베딩
##  문장을 의미있게 분석하는 기술 중 성능이 좋은 기술이 트랜스포머
##  팁은 우선은 개념만 이해하면서 소스를 이해하시는거 추천..
##  트랜스포머 기술로 만든 모델이 있습니다. BERT 종류..
##  이런 모델을 허깅페이스에 많이 올려 놓고
##  올린 사람이 완전 무료 오픈으로 풀던가, 승인 받고써라. 운영
##  임베딩을 하면서 동시에 데이터베이스 저장.. 데이터베이스는  FAISS
vectorstore = FAISS.from_documents(docs, embeddings)

In [6]:
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma
from langchain_community.embeddings import HuggingFaceEmbeddings

# 1. 임베딩 모델 생성
#embeddings = OpenAIEmbeddings()

embeddings = HuggingFaceEmbeddings(
    model_name="jhgan/ko-sroberta-multitask",
    model_kwargs={"device": "cpu"},  # GPU 있으면 "cuda"
    encode_kwargs={"normalize_embeddings": True} # 벡터 비교에서 중요한 건 “방향”임
)


C:\Users\Public\Documents\ESTsoft\CreatorTemp\ipykernel_3484\4265017485.py:8: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(


In [7]:

# 2. Chroma 벡터 DB 생성
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory="../rag/chroma_db_jhgan"
)

# documents=docs: 저장할 문서
# embedding=embeddings: 문서를 벡터로 변환할 모델
# persist_directory="./chroma_db": 벡터 DB를 저장할 폴더

In [8]:
print("저장된 문서 개수:", vectorstore._collection.count())

저장된 문서 개수: 194


In [9]:
## 저장된 문서 불러오기
from langchain_chroma import Chroma
vectorstore = Chroma(
    persist_directory="../rag/chroma_db_jhgan",
    embedding_function=embeddings
)

In [25]:
## 유사도 검색하기
query = "자장면 음식 소개해줘?"

results = vectorstore.similarity_search(
    query,
    k=3
)

for doc in results:
    print(doc.page_content)

이름:양장피
    설명:해산물과 채소를 겨자소스에 곁들인 중국 요리
이름:콩국수
    설명:고소한 콩물에 면을 넣어 먹는 음식
이름:칼국수
    설명:넓적한 밀가루 면을 국물에 끓인 음식


In [10]:
# =========================
# 5. Retrieval 함수 (점수 포함)
# 0.0	완전히 동일
# 0.2	매우 유사
# 1.0	보통
# 5.0	전혀 안 비슷
# =========================
def search_travel(query, k=3):
    results = vectorstore.similarity_search_with_score(query, k=k)

    output = []
    for doc, score in results:
        output.append({
            # "질문": doc.metadata["query"],
            # "여행지": doc.metadata["destination"],
            # "위험도": doc.metadata["risk_level"],
            # "위험이유": doc.metadata["risk_reason"],
            "내용": doc.page_content,
            "유사도": score
        })

    # score는 보통 낮을수록 유사 (FAISS 기준)
    output = sorted(output, key=lambda x: x["유사도"])
    # FAISS 사용 중

    return output


# =========================
# 6. 테스트
# =========================
# query = "유럽에서 대중 교통이 편리한 여행지 소개 해 줘"
query = "만두를 바삭하게 구운 음식 소개해 줘"
results = search_travel(query)
print(results)
for r in results:
    print("유사도:", r["유사도"])
    # print("📍 여행지:", r["여행지"])
    # print("❓ 질문:", r["질문"])
    # print("⚠️ 위험도:", r["위험도"])
    print(r["내용"])
    print("-" * 50)

[{'내용': '이름:군만두\n    설명:속을 채운 만두를 바삭하게 구운 음식', '유사도': 0.3369998335838318}, {'내용': '이름:군만두\n    설명:속을 채운 만두를 바삭하게 구운 음식', '유사도': 0.3369998335838318}, {'내용': '이름:만둣국\n    설명:만두를 따뜻한 국물에 넣어 끓인 음식', '유사도': 0.6472766995429993}]
유사도: 0.3369998335838318
이름:군만두
    설명:속을 채운 만두를 바삭하게 구운 음식
--------------------------------------------------
유사도: 0.3369998335838318
이름:군만두
    설명:속을 채운 만두를 바삭하게 구운 음식
--------------------------------------------------
유사도: 0.6472766995429993
이름:만둣국
    설명:만두를 따뜻한 국물에 넣어 끓인 음식
--------------------------------------------------


In [12]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv

load_dotenv()
llm = ChatOpenAI(model="gpt-4o-mini")

In [17]:
## 지금까지는 문장을 로딩해서 Document를 만들고 벡터데이터베이스 구축
## 이제는 랭체인을 만들겁니다.
## 질의 > 유사도가 높은 2개의 정보를 가져옵니다. >
## 사용자가 입력한 질문과, 유사도 검색을 통해 가져온 정보를 모두
## LLM에게 전달하고. 역할과 조건을 지시하여 원하는 답을 생성해 달라.

## 랭체인을 구성하기 위한 객체?

## 템플릿(llm에게 질문을 정규화) 만들어서 입력 받은 값과 매핑하려고
from langchain_core.prompts import ChatPromptTemplate
## 랭체인에서는 runable상태를 유지해야 한다.. 그래서 사용한다.
## 그냥 함수를 처리하면 runable상태가 아니라서 RunnableLambda로 처리
## runable상태는 입력이 있고 처리를 하고 출력을 다음에 넘겨준다.
from langchain_core.runnables import RunnableLambda
## llm답변의 타입을 지정.. StrOutputParser 문자열로 대답해 줘
from langchain_core.output_parsers import StrOutputParser

## RAG
## Retrieval (검색) - 벡터베이스 베이스.. 이미 구축
## R검색해서 유사도가 높은 순으로 데이터를 가져오겠다
def retrieval_docs(query):
    results = vectorstore.similarity_search(query,k=3)
    return results
## Augmented (보강)
## A 검색한 정보를 모델에 입력하기 위해서 보강하겠다
def format_docs(docs):
    #retrieval_docs 결과는 리스트인 결과를 합쳐야 한다.
    # data=''
    # for doc_data in docs:
    #     #print(doc_data)
    #     data = data + '\n\n' +doc_data.page_content
    # return data
    return "\n\n".join([doc_data.page_content for doc_data in docs])
## 테스트 코드
# testq="해외 유럽여행 중 조용한 곳"
# t = retrieval_docs(testq)
# t1 =format_docs(t)
# print(t1)


## Generation (생성)
## G 답변을 생각해라.
prompt = ChatPromptTemplate.from_template("""
너는 음식 추천 전문가야. 참고 정보를 이용해서 음식별로 1. 2. 형식으로 대답 
[사용자질문] 
{query}
[참고정보]
{context} 
[규칙]
- 제공한 참고정보만으로 답변을 만들 것
- 음식이름을 먼저 이야기 해 
- 2줄 정도 요약해 줘
- 친절하게 구매할 수 있도록 이야기 해 줘                            
""")

## RAG  chain
rag_chain = (
    {
        "query": lambda x : x['query'], #사용자의 입력은 딕셔너리라는 뜻
        "context": 
        RunnableLambda(lambda x : format_docs(retrieval_docs(x['query'])))
    }
    |prompt  # 입력변수가 몇개 인지 분석해서 이전의 단계에서 자료형 정의시 참고
    |llm
    |StrOutputParser()
)   
result = rag_chain.invoke({'query':"만두를 바삭하게 구운 음식 소개해 줘"})
print("LLM이 생성한 답입니다.")
print(result)

LLM이 생성한 답입니다.
1. 군만두  
   속을 채운 만두를 바삭하게 구운 음식입니다. 바삭한 식감이 일품인 군만두는 다양한 소스와 함께 즐길 수 있어 더욱 맛있습니다. 가까운 주점이나 중국 음식점에서 쉽게 구매할 수 있습니다. 

2. 만둣국  
   만두를 따뜻한 국물에 넣어 끓인 음식입니다. 국물의 깊은 맛과 만두의 조화가 매력적인 만둣국은 집에서 손쉽게 곁들여 먹을 수 있는 메뉴입니다. 슈퍼마켓에서 만두를 사서 간편하게 요리해 보세요.


In [43]:
import sys
import langchain_core

print("Python 경로:", sys.executable)
print("LangChain Core 버전:", langchain_core.__version__)

Python 경로: d:\dev\2026\js_fastApI\.venv\Scripts\python.exe
LangChain Core 버전: 1.7.6


In [35]:
import sys

print("Python 경로:", sys.executable)

import langchain
import langchain_core
import langchain_openai
import langchain_community

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma

print("LangChain import 성공!")
print("langchain:", langchain.__version__)
print("langchain_core:", langchain_core.__version__)

Python 경로: d:\dev\2026\js_fastApI\.venv\Scripts\python.exe
LangChain import 성공!
langchain: 0.3.27
langchain_core: 1.7.6
